In [1]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [4]:
from transformers import pipeline
classifier = pipeline("text-classification", 
                        model="j-hartmann/emotion-english-distilroberta-base", 
                        top_k=None,
                        device = "mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764603149145842},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138523945584893}]]

In [5]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [6]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.6548408269882202},
  {'label': 'neutral', 'score': 0.16985228657722473},
  {'label': 'sadness', 'score': 0.11640898138284683},
  {'label': 'surprise', 'score': 0.02070062980055809},
  {'label': 'disgust', 'score': 0.019100766628980637},
  {'label': 'joy', 'score': 0.015161330811679363},
  {'label': 'anger', 'score': 0.003935159184038639}]]

In [9]:
sentences = books["description"][0].split(".")
predictions = classifier(books["description"][0].split("."))
predictions

[[{'label': 'surprise', 'score': 0.7296028733253479},
  {'label': 'neutral', 'score': 0.14038552343845367},
  {'label': 'fear', 'score': 0.0681622177362442},
  {'label': 'joy', 'score': 0.04794244095683098},
  {'label': 'anger', 'score': 0.009156350046396255},
  {'label': 'disgust', 'score': 0.002628473099321127},
  {'label': 'sadness', 'score': 0.0021221598144620657}],
 [{'label': 'neutral', 'score': 0.449371874332428},
  {'label': 'disgust', 'score': 0.2735903263092041},
  {'label': 'joy', 'score': 0.1090828999876976},
  {'label': 'sadness', 'score': 0.09362737089395523},
  {'label': 'anger', 'score': 0.04047823324799538},
  {'label': 'surprise', 'score': 0.026970192790031433},
  {'label': 'fear', 'score': 0.006879065185785294}],
 [{'label': 'neutral', 'score': 0.6462162137031555},
  {'label': 'sadness', 'score': 0.2427331507205963},
  {'label': 'disgust', 'score': 0.043422602117061615},
  {'label': 'surprise', 'score': 0.0283005740493536},
  {'label': 'joy', 'score': 0.0142114423215

So the idea is to add these labels as seperate columns and then find the max probablilty for each label to be assigned to the book description 

In [10]:
sorted(predictions[0], key= lambda x: x["label"])

[{'label': 'anger', 'score': 0.009156350046396255},
 {'label': 'disgust', 'score': 0.002628473099321127},
 {'label': 'fear', 'score': 0.0681622177362442},
 {'label': 'joy', 'score': 0.04794244095683098},
 {'label': 'neutral', 'score': 0.14038552343845367},
 {'label': 'sadness', 'score': 0.0021221598144620657},
 {'label': 'surprise', 'score': 0.7296028733253479}]

In [11]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [12]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [13]:
emotion_scores

{'anger': [np.float64(0.06413350999355316),
  np.float64(0.6126185059547424),
  np.float64(0.06413350999355316),
  np.float64(0.35148298740386963),
  np.float64(0.08141236007213593),
  np.float64(0.23222509026527405),
  np.float64(0.5381847620010376),
  np.float64(0.06413350999355316),
  np.float64(0.30067047476768494),
  np.float64(0.06413350999355316)],
 'disgust': [np.float64(0.2735903263092041),
  np.float64(0.34828588366508484),
  np.float64(0.10400654375553131),
  np.float64(0.15072251856327057),
  np.float64(0.1844949722290039),
  np.float64(0.7271745800971985),
  np.float64(0.1558549851179123),
  np.float64(0.10400654375553131),
  np.float64(0.27948084473609924),
  np.float64(0.17792603373527527)],
 'fear': [np.float64(0.928168535232544),
  np.float64(0.9425276517868042),
  np.float64(0.9723208546638489),
  np.float64(0.3607071340084076),
  np.float64(0.09504334628582001),
  np.float64(0.05136271193623543),
  np.float64(0.7474280595779419),
  np.float64(0.404496431350708),
  np

In [14]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [04:58<00:00, 17.44it/s]


In [17]:
emotion_df= pd.DataFrame(emotion_scores)
emotion_df["isbn13"] = isbn
emotion_df.head(15)

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273590,0.928169,0.932798,0.646216,0.967158,0.729603,9780002005883
1,0.612619,0.348286,0.942528,0.704422,0.887939,0.111690,0.252545,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765,9780006178736
3,0.351483,0.150723,0.360707,0.251881,0.732687,0.111690,0.078765,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.884389,0.475881,0.078765,9780006280934
5,0.232225,0.727175,0.051363,0.043376,0.621393,0.111690,0.271903,9780006380832
6,0.538185,0.155855,0.747428,0.872565,0.712194,0.407999,0.078765,9780006470229
7,0.064134,0.104007,0.404496,0.040564,0.549477,0.820282,0.234488,9780006472612
8,0.300670,0.279481,0.915524,0.040564,0.840290,0.354459,0.135615,9780006482079
9,0.064134,0.177926,0.051363,0.040564,0.860372,0.111690,0.078765,9780006483014


In [18]:
books = pd.merge(books, emotion_df, on="isbn13")
books.head(15)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273590,0.928169,0.932798,0.646216,0.967158,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612619,0.348286,0.942528,0.704422,0.887939,0.111690,0.252545
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351483,0.150723,0.360707,0.251881,0.732687,0.111690,0.078765
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.884389,0.475881,0.078765
5,9780006380832,0006380832,Empires of the Monsoon,Richard Hall,"Africa, East",http://books.google.com/books/content?id=MuPEQ...,Until Vasco da Gama discovered the sea-route t...,1998.0,4.41,608.0,...,Empires of the Monsoon: A History of the India...,9780006380832 Until Vasco da Gama discovered t...,Nonfiction,0.232225,0.727175,0.051363,0.043376,0.621393,0.111690,0.271903
6,9780006470229,000647022X,The Gap Into Madness,Stephen R. Donaldson,"Hyland, Morn (Fictitious character)",http://books.google.com/books/content?id=4oXav...,A new-cover reissue of the fourth book in the ...,1994.0,4.15,743.0,...,The Gap Into Madness: Chaos and Order,9780006470229 A new-cover reissue of the fourt...,Fiction,0.538185,0.155855,0.747428,0.872565,0.712194,0.407999,0.078765
7,9780006472612,0006472613,Master of the Game,Sidney Sheldon,Adventure stories,http://books.google.com/books/content?id=TkTYp...,Kate Blackwell is an enigma and one of the mos...,1982.0,4.11,489.0,...,Master of the Game,9780006472612 Kate Blackwell is an enigma and ...,Nonfiction,0.064134,0.104007,0.404496,0.040564,0.549477,0.820282,0.234488
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,"Tricked once more by his wily half-brother, Ly...",1995.0,4.03,522.0,...,Warhost of Vastmark,9780006482079 Tricked once more by his wily ha...,Fiction,0.300670,0.279481,0.915524,0.040564,0.840290,0.354459,0.135615
9,9780006483014,0006483011,The Once and Future King,Terence Hanbury White,Arthurian romances,http://books.google.com/books/content?id=Jx6Bv...,An omnibus volume of the author's complete sto...,1996.0,4.04,823.0,...,The Once and Future King,9780006483014 An omnibus volume of the author'...,Fiction,0.064134,0.177926,0.051363,0.040564,0.860372,0.111690,0.078765


In [20]:
books.to_csv("books_with_emotions.csv", index=False)